In [ ]:
!pip install ipywidgets --upgrade

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
# test_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
# train_df.describe(include='object')
# test_df.describe(include='object')

In [ ]:
df['input'] =  '; TEXT2: ' + df.text

In [ ]:
from datasets import Dataset,DatasetDict

ds = Dataset.from_pandas(df)
ds

In [ ]:
model_nm = 'microsoft/deberta-v3-small'

In [ ]:
from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_nm)

In [ ]:
def tok_func(x): return tokz(x["input"])

In [ ]:
tok_ds = ds.map(tok_func, batched=True)

In [ ]:
vocab_size = len(tokz.vocab)
print(f"The size of the vocabulary list is: {vocab_size}")

In [ ]:
tok_ds = tok_ds.rename_columns({'target':'labels'})

In [ ]:
eval_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
eval_df.describe()

In [ ]:
eval_df['input'] = 'TEXT2: ' + eval_df.text 
eval_ds = Dataset.from_pandas(eval_df).map(tok_func, batched=True)

In [ ]:
print(eval_df.head())

In [ ]:
dds = tok_ds.train_test_split(0.25, seed=42)
dds

In [ ]:
# def corr_d(eval_pred): return {'pearson': corr(*eval_pred)}

In [ ]:
from transformers import TrainingArguments,Trainer
bs = 32
epochs = 1
lr = 8e-5

In [ ]:
args = TrainingArguments('outputs', learning_rate=lr, warmup_ratio=0.1, lr_scheduler_type='cosine', fp16=True,
    eval_strategy="epoch", per_device_train_batch_size=bs, per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs, weight_decay=0.01, report_to='none')

In [ ]:
from sklearn.metrics import f1_score
def my_metric(eval_pred): 
    return {'F1': f1_score(*eval_pred, zero_division=0)}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2)
trainer = Trainer(model, args, train_dataset=dds['train'], eval_dataset=dds['test'],
                  tokenizer=tokz)

In [ ]:
trainer.train();

In [ ]:
preds = trainer.predict(eval_ds)
preds

In [ ]:
preds.predictions

In [ ]:
class_preds = np.argmax(preds.predictions, axis=-1)
class_preds

In [ ]:
sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
sub['target'] = class_preds
sub

In [ ]:
name = os.path.basename(model_nm)
sub.to_csv(f'submission.csv', index=False)